# Datasets Sources
- https://www.kaggle.com/competitions/demand-forecasting-kernels-only/models
- https://www.kaggle.com/competitions/m5-forecasting-accuracy/data


# Model
- LSTM, GRU, RNN comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import random
import time
import json
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Hyperparameter Setup
BATCH_SIZE = 64
EPOCHS = 15
learning_rate = 1e-3
SEED = 7
LOOKBACK = 60
HORIZON = 28                 # ตั้งเป็น 1 ได้เพื่อดูอาการทำนายตามหลังแบบวันต่อวัน
HIDDEN_SIZE = 32
MODEL_NAMES = ['LSTM', 'GRU', 'RNN']  # หรือ ['GRU'] เพื่อเริ่มรันตัวเดียว
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15


DATASET = 'm5'
M5_SERIES_ID = None         # None = แถวแรก; หรือใส่ id ที่มีในไฟล์ M5
ROOT = Path('/content/drive/MyDrive/demand-forecasting')
RUN_NAME = 'm5_model_comparison_v2'  # เปลี่ยนชื่อเมื่อทำการทดลองใหม่
OUTPUT_DIR = ROOT / 'outputs' / DATASET / RUN_NAME
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
assert LOOKBACK >= 7 and HORIZON >= 1 and EPOCHS >= 1
assert 0 < TRAIN_RATIO < 1 and 0 < VAL_RATIO < 1 - TRAIN_RATIO
assert MODEL_NAMES and len(set(MODEL_NAMES)) == len(MODEL_NAMES)
assert set(MODEL_NAMES).issubset({'GRU', 'LSTM', 'RNN'})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', device, '| PyTorch:', torch.__version__)

In [ ]:
# Build M5 Time-Series Pipeline
path = ROOT / 'data/m5/sales_train_evaluation.csv'
# อ่านเป็น chunk เพื่อไม่โหลดทุกสินค้าเข้า RAM
selected = None
for chunk in pd.read_csv(path, chunksize=100):
    found = chunk.iloc[:1] if M5_SERIES_ID is None else chunk[chunk['id'] == M5_SERIES_ID]
    if not found.empty:
        selected = found.iloc[0]
        break
if selected is None:
    raise ValueError('ไม่พบ M5_SERIES_ID ในไฟล์')
day_cols = sorted([c for c in selected.index if c.startswith('d_')],
                  key=lambda c: int(c[2:]))
calendar = pd.read_csv(ROOT / 'data/m5/calendar.csv', usecols=['d', 'date'])
assert not calendar['d'].duplicated().any(), 'รหัสวันใน calendar ซ้ำ'
raw = pd.DataFrame({
    'date': pd.to_datetime(calendar.set_index('d').loc[day_cols, 'date']).to_numpy(),
    'sales': selected[day_cols].to_numpy(dtype=np.float32),
})
series_name = str(selected['id'])

assert not raw.empty, 'ไม่พบสินค้า/ร้านที่เลือก'
assert not raw[['date', 'sales']].isna().any().any(), 'ข้อมูลหาย ต้องตรวจต้นทางก่อน'
assert not raw['date'].duplicated().any(), 'วันที่ซ้ำ'
assert pd.DatetimeIndex(raw['date']).equals(pd.date_range(raw.date.iloc[0], raw.date.iloc[-1])), 'วันที่ขาดหาย'
assert np.isfinite(raw['sales'].to_numpy(dtype=float)).all() and (raw['sales'] >= 0).all()
sales = raw['sales'].to_numpy(dtype=np.float32)
dates = pd.DatetimeIndex(raw['date'])
n = len(sales)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))
assert train_end >= LOOKBACK + HORIZON + 1, 'ประวัติ Train สั้นเกินไป'
assert val_end - train_end >= HORIZON and n - val_end >= HORIZON

split_table = pd.DataFrame([
    {'split':'Train', 'start':str(dates[0].date()), 'end':str(dates[train_end-1].date())},
    {'split':'Validation', 'start':str(dates[train_end].date()), 'end':str(dates[val_end-1].date())},
    {'split':'Test', 'start':str(dates[val_end].date()), 'end':str(dates[-1].date())},
])
print(series_name, '| days:', n)
display(raw.head())
display(split_table)

In [ ]:
# Explore the time series
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dates, sales, linewidth=0.8)
ax.axvline(dates[train_end], color='orange', linestyle='--', label='Validation starts')
ax.axvline(dates[val_end], color='red', linestyle='--', label='Test starts')
ax.set(title=series_name, xlabel='Date', ylabel='Units sold')
ax.legend()
plt.show()

In [ ]:
# Scaling and normalization


In [ ]:
# Build Dataloader


In [ ]:
# Build Model


In [ ]:
# Inspect shapes

In [ ]:
# Define loss and optimizer

In [ ]:
# Training and Validating

In [ ]:
# Plot learning curves

In [ ]:
# Evaluation